<a href="https://colab.research.google.com/github/Navendu13/apriori-alpha-project/blob/main/apriori_game_theoretic_alpha_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q yfinance

## Step 1: Data Acquisition — Building the Research Universe

**What this step does:**
This cell downloads 11+ years (Jan 2015 – Jul 2026) of daily OHLCV (Open, High, Low, Close, Volume)
data for a 30-stock cross-sector universe using the `yfinance` API, then extracts and saves clean
close price and volume matrices as CSV files.

**Why we are doing this:**
Every downstream step (signal construction, hypothesis testing, backtesting) depends entirely on
having clean, aligned price and volume data. Without a properly structured dataset, no signal or
statistical test can be trusted.

**Why this specific universe (30 stocks, cross-sector):**
We deliberately chose a diversified basket spanning Tech, Financials, Healthcare, Energy, and
Consumer sectors rather than a single sector. This is a methodological safeguard: if a signal only
works in one sector, it's likely a sector-specific fluke rather than a genuine market microstructure
effect. A signal that holds across sectors is far more credible and defensible.

**Why this specific timeframe (2015–2026):**
This window intentionally spans multiple market regimes — the 2015-2019 low-volatility bull run,
the 2020 COVID crash and recovery, the 2022 rate-hike bear market, and the 2023-2026 recovery.
Testing across regimes is a standard robustness check; a signal that only works in one type of
market environment is not reliable.

**What happens if we skip or shortcut this step:**
Any signal or backtest built on incomplete, misaligned, or narrow data (e.g., only bull-market years,
or only tech stocks) risks being overfit or non-generalizable — a critical flaw that any technically
rigorous reviewer (like a quant fund CTO) would immediately flag.

In [3]:
import yfinance as yf
import pandas as pd, os

# Diversified, liquid, cross-sector universe — avoids single-sector overfit criticism
universe = [
    'AAPL','MSFT','NVDA','GOOGL','AMZN','META','AVGO','TSLA',   # Tech/Comm
    'JPM','BAC','GS','MS',                                       # Financials
    'UNH','JNJ','PFE','LLY',                                     # Healthcare
    'XOM','CVX','COP',                                           # Energy
    'PG','KO','PEP','WMT','COST',                                # Consumer staples
    'HD','DIS','NFLX','ADBE','CRM','V'                           # Consumer disc./services
]

# 2015–2026 spans multiple regimes: 2015-19 bull, 2020 COVID crash/recovery,
# 2022 rate-hike bear market, 2023-26 recovery — needed for walk-forward robustness
start_date = '2015-01-01'
end_date = '2026-07-29'

data = yf.download(universe, start=start_date, end=end_date,
                    group_by='ticker', auto_adjust=True, progress=False)

os.makedirs('output', exist_ok=True)

close_prices = pd.concat({t: data[t]['Close'] for t in universe
                           if t in data.columns.get_level_values(0)}, axis=1)
volumes = pd.concat({t: data[t]['Volume'] for t in universe
                      if t in data.columns.get_level_values(0)}, axis=1)

close_prices.to_csv('output/close_prices.csv')
volumes.to_csv('output/volumes.csv')

print(close_prices.shape, volumes.shape)
close_prices.tail()

(2908, 30) (2908, 30)


,AAPL,MSFT,NVDA,GOOGL,AMZN,META,AVGO,TSLA,JPM,BAC,...,KO,PEP,WMT,COST,HD,DIS,NFLX,ADBE,CRM,V
Date,,,,,,,,,,,,,,,,,,,,,
2026-07-22,325.890015,390.339996,212.059998,342.089996,244.850006,627.169983,396.809998,374.010010,348.209991,61.619999,...,82.199997,135.649994,109.330002,925.838013,331.450012,95.870003,68.529999,218.360001,163.000000,353.420013
2026-07-23,321.660004,381.579987,208.759995,317.690002,233.660004,606.099976,392.470001,319.690002,349.899994,61.279999,...,81.169998,134.949997,108.400002,924.589966,324.709991,92.830002,68.889999,212.169998,156.929993,351.600006
2026-07-24,333.019989,381.700012,206.839996,319.739990,232.110001,595.190002,381.920013,313.029999,353.209991,62.049999,...,82.250000,136.639999,109.470001,935.030029,332.980011,94.849998,70.089996,225.110001,163.660004,355.739990
2026-07-27,336.910004,389.100006,196.509995,326.559998,231.389999,593.869995,383.220001,309.220001,356.200012,62.130001,...,84.070000,139.789993,111.739998,951.580017,336.089996,96.650002,70.400002,237.750000,173.600006,362.529999
2026-07-28,340.079987,393.350006,197.009995,333.709991,230.860001,593.409973,380.910004,307.440002,357.309998,62.619999,...,88.269997,142.860001,113.099998,966.580017,344.470001,98.889999,72.389999,249.179993,181.500000,366.589996


## Step 1 Results: Data Validation

**What the output shows:**
The pull returned a (2908, 30) matrix for both close prices and volumes — meaning 2,908 trading
days across all 30 tickers, with no missing tickers, confirming a complete and aligned dataset.

**Interpreting the numbers:**
2,908 trading days is consistent with ~11.5 years of data (roughly 252 trading days/year), which
matches our intended 2015–2026 window. The tail rows show plausible, realistic closing prices
(e.g., AAPL ~$340, NVDA ~$197) with no NaNs or broken values visible.

**Is this realistic enough to proceed:**
Yes. Matching row/column counts across both price and volume matrices confirms no silent data
loss or misalignment occurred during the download and merge process.

**How this feeds into the next step:**
This clean price/volume matrix is the direct input for VPIN signal construction (Step 2). Any
gaps or misalignment here would propagate errors into every later calculation, so validating
shape and sanity now prevents having to debug much more confusing errors later in the pipeline.

## Step 2: Constructing the VPIN Signal (Game-Theoretic Order Flow Proxy)

**What this step does:**
This cell computes VPIN (Volume-Synchronized Probability of Informed Trading) for each stock,
using daily returns and volume as inputs. It classifies each day's volume into estimated
buy-initiated vs. sell-initiated portions using a z-score-based Bulk Volume Classification (BVC)
method, then computes a rolling order-flow imbalance ratio.

**Why we are doing this:**
VPIN is grounded in the Easley-O'Hara sequential trade model, a game-theoretic framework where
market makers (uninformed) and informed traders interact strategically. Market makers cannot
directly observe who is informed, so they infer risk from the imbalance of buy vs. sell order
flow. This directly aligns with A Priori's stated focus on "game theory" — it is not a generic
technical indicator, but a signal rooted in strategic market microstructure theory.

**Why we approximate with BVC instead of true tick-level classification:**
The gold-standard method (Lee-Ready algorithm) requires tick-by-tick trade and quote data, which
we don't have. BVC is the accepted academic substitute when only daily OHLCV is available — it
infers buy/sell pressure from how much price moved relative to recent volatility.

**What happens if we skip this step:**
Without VPIN (or an equivalent microstructure-based proxy), the project would just be a standard
price/volume technical analysis exercise, with no genuine connection to game theory — undermining
the core differentiation strategy for this pitch.

In [4]:
import pandas as pd, numpy as np, os

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vol = pd.read_csv('output/volumes.csv', index_col=0, parse_dates=True)

def compute_vpin(price, volume, bucket_size_frac=0.02, window=50):
    ret = price.pct_change()
    daily_vol = volume
    total_vol = daily_vol.sum()
    bucket_size = total_vol * bucket_size_frac
    sigma = ret.rolling(20).std()
    z = (ret / sigma).clip(-5, 5)
    from scipy.stats import norm
    buy_frac = norm.cdf(z.fillna(0))
    buy_vol = daily_vol * buy_frac
    sell_vol = daily_vol * (1 - buy_frac)
    imbalance = (buy_vol - sell_vol).abs()
    vpin = imbalance.rolling(window).sum() / daily_vol.rolling(window).sum()
    return vpin

vpin_df = pd.DataFrame(index=close.index)
for t in close.columns:
    vpin_df[t] = compute_vpin(close[t], vol[t])

vpin_df.to_csv('output/vpin_signal.csv')
print(vpin_df.shape)
print(vpin_df.tail(3))
print(vpin_df.describe().T[['mean','std','min','max']].head(10))

(2908, 30)
                AAPL      MSFT      NVDA     GOOGL      AMZN      META  \
Date                                                                     
2026-07-24  0.550279  0.572202  0.515895  0.515387  0.579539  0.529870   
2026-07-27  0.547598  0.579197  0.521924  0.510596  0.569668  0.521824   
2026-07-28  0.550328  0.579532  0.504956  0.517625  0.561183  0.520117   

                AVGO      TSLA       JPM       BAC  ...        KO       PEP  \
Date                                                ...                       
2026-07-24  0.551437  0.557421  0.542339  0.495780  ...  0.535317  0.520605   
2026-07-27  0.549724  0.543886  0.538618  0.481685  ...  0.545904  0.521390   
2026-07-28  0.538352  0.544077  0.541207  0.490725  ...  0.567102  0.529319   

                 WMT      COST        HD       DIS      NFLX      ADBE  \
Date                                                                     
2026-07-24  0.496395  0.532467  0.531131  0.500173  0.517117  0.554927   


## Step 2 Results: VPIN Signal Validation

**What the output shows:**
VPIN values across all 30 stocks average around 0.51–0.53, with standard deviations of roughly
0.04–0.05, and a full range spanning approximately 0.20 to 0.76.

**Interpreting the numbers:**
VPIN is bounded between 0 and 1 by construction. A value near 0.5 indicates balanced buy/sell
volume (normal, uninformed trading conditions). Values pushing toward 0.6–0.76 indicate periods
where volume is heavily skewed to one side — a signal of potentially elevated informed trading
or order flow "toxicity."

**Is this realistic enough to proceed:**
Yes. This distribution (centered near 0.5, with moderate spread and occasional extremes) is
consistent with published VPIN studies in the market microstructure literature. If our values had
clustered near 0 or 1, or shown no variation at all, that would indicate a bug in the calculation.

**How this feeds into the next step:**
At this stage, VPIN is just a number sitting next to price history — it has no proven predictive
value yet. Step 3 formally tests whether elevated VPIN actually precedes unusual returns or
volatility, which determines whether this signal is a real, usable input or just noise.

## Step 3: Hypothesis Testing — Does VPIN Actually Predict Anything?

**What this step does:**
This cell tests whether high-VPIN days are followed by statistically different forward returns
(1-day, 5-day) and forward volatility (5-day) compared to low-VPIN days. For each stock, VPIN is
ranked into quintiles based on its own historical distribution, and a Welch's t-test compares the
extreme (top vs. bottom) quintiles.

**Why we are doing this:**
A signal is only useful if it has demonstrated, statistically significant predictive power —
otherwise it's just an interesting-looking number with no practical value. This step is the
critical "does it actually work" checkpoint before any strategy is built on top of VPIN.

**Why we shift returns/volatility forward (no lookahead bias):**
Forward returns and volatility are computed using `.shift(-1)` and `.shift(-5)`, ensuring VPIN on
day t is only ever compared to information that occurs strictly after day t. This prevents
lookahead bias — a critical, commonly-tested error in quant interviews, where future information
accidentally leaks into a predictive signal, producing falsely inflated results.

**Why we test across all 30 stocks instead of just one:**
Testing a signal on a single stock risks finding a coincidental pattern (overfitting/noise).
Requiring consistent results across a diversified 30-stock universe is a standard robustness
check that distinguishes a genuine market effect from a fluke.

**What happens if we skip this step:**
Building a trading strategy directly on VPIN without this validation step would risk deploying
a signal that is pure noise, dressed up as insight — exactly the kind of unvalidated claim that a
quant reviewer would immediately reject.

In [5]:
import pandas as pd, numpy as np
from scipy import stats

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)

fwd_ret_1d = close.pct_change().shift(-1)
fwd_ret_5d = close.pct_change(5).shift(-5)
fwd_vol_5d = close.pct_change().rolling(5).std().shift(-5)

results = []
for t in close.columns:
    df = pd.DataFrame({
        'vpin': vpin[t],
        'fwd_ret_1d': fwd_ret_1d[t],
        'fwd_ret_5d': fwd_ret_5d[t],
        'fwd_vol_5d': fwd_vol_5d[t]
    }).dropna()
    if len(df) < 200:
        continue
    df['quintile'] = pd.qcut(df['vpin'], 5, labels=False, duplicates='drop')
    low = df[df['quintile'] == 0]
    high = df[df['quintile'] == df['quintile'].max()]
    t_ret1, p_ret1 = stats.ttest_ind(high['fwd_ret_1d'], low['fwd_ret_1d'], equal_var=False)
    t_ret5, p_ret5 = stats.ttest_ind(high['fwd_ret_5d'], low['fwd_ret_5d'], equal_var=False)
    t_vol5, p_vol5 = stats.ttest_ind(high['fwd_vol_5d'], low['fwd_vol_5d'], equal_var=False)
    results.append({
        'ticker': t,
        'low_vpin_fwdret1d': low['fwd_ret_1d'].mean(),
        'high_vpin_fwdret1d': high['fwd_ret_1d'].mean(),
        'p_ret1d': p_ret1,
        'low_vpin_fwdret5d': low['fwd_ret_5d'].mean(),
        'high_vpin_fwdret5d': high['fwd_ret_5d'].mean(),
        'p_ret5d': p_ret5,
        'low_vpin_fwdvol5d': low['fwd_vol_5d'].mean(),
        'high_vpin_fwdvol5d': high['fwd_vol_5d'].mean(),
        'p_vol5d': p_vol5,
    })

res_df = pd.DataFrame(results)
res_df.to_csv('output/vpin_hypothesis_test.csv', index=False)

print("=== Volatility prediction (high VPIN vs low VPIN, 5-day forward vol) ===")
print(f"Stocks where high VPIN -> higher fwd vol: {(res_df['high_vpin_fwdvol5d'] > res_df['low_vpin_fwdvol5d']).sum()} / {len(res_df)}")
print(f"Stocks with significant vol diff (p<0.05): {(res_df['p_vol5d'] < 0.05).sum()} / {len(res_df)}")
print()
print("=== Return prediction (5-day) ===")
print(f"Stocks with significant return diff (p<0.05): {(res_df['p_ret5d'] < 0.05).sum()} / {len(res_df)}")
print()
print(res_df[['ticker','low_vpin_fwdvol5d','high_vpin_fwdvol5d','p_vol5d','p_ret5d']].round(4).to_string(index=False))

=== Volatility prediction (high VPIN vs low VPIN, 5-day forward vol) ===
Stocks where high VPIN -> higher fwd vol: 30 / 30
Stocks with significant vol diff (p<0.05): 28 / 30

=== Return prediction (5-day) ===
Stocks with significant return diff (p<0.05): 7 / 30

ticker  low_vpin_fwdvol5d  high_vpin_fwdvol5d  p_vol5d  p_ret5d
  AAPL             0.0119              0.0203   0.0000   0.0009
  MSFT             0.0129              0.0181   0.0000   0.1512
  NVDA             0.0228              0.0318   0.0000   0.0045
 GOOGL             0.0139              0.0199   0.0000   0.7272
  AMZN             0.0164              0.0191   0.0001   0.8148
  META             0.0205              0.0212   0.4125   0.0455
  AVGO             0.0217              0.0241   0.0100   0.8755
  TSLA             0.0305              0.0328   0.0535   0.1489
   JPM             0.0110              0.0180   0.0000   0.6218
   BAC             0.0128              0.0212   0.0000   0.4276
    GS             0.0145        

## Step 3 Results: VPIN Predicts Volatility, Not Direction

**What the output shows:**
Across all 30 stocks, high-VPIN days were followed by higher 5-day forward volatility in 30/30
cases, with 28/30 statistically significant at p<0.05. In contrast, only 7/30 stocks showed a
statistically significant difference in forward *returns* between high and low VPIN quintiles.

**Interpreting the numbers:**
A p-value below 0.05 means there is less than a 5% probability the observed difference occurred
by random chance — the conventional statistical significance threshold. 28/30 significant results
for volatility is an unusually strong and consistent finding; 7/30 for returns is close to what
you'd expect from random chance alone (about 1-2 out of 30 by pure chance at the 5% threshold, so
7/30 suggests a weak but not fully random directional effect).

**Is this realistic enough to accept:**
Yes — and importantly, this result is *more* credible because it is not a suspiciously perfect
"free money" signal. VPIN measuring order-flow imbalance should theoretically predict volatility
(market makers widening spreads, liquidity thinning) rather than direction, since informed traders
can be informed about either upside or downside news. This aligns with the underlying theory,
which increases confidence the result is genuine rather than a statistical artifact.

**How this reshapes the project going forward:**
This finding redirects the project from "VPIN as a directional alpha signal" (which the data does
not support) to "VPIN as a volatility-timing / risk-management overlay" (which the data strongly
supports) — e.g., reducing position size or tightening risk controls ahead of high-VPIN periods.
This is a more defensible, theory-consistent, and practically useful application, and it directly
sets up the next step: designing a risk-managed strategy that uses VPIN as a volatility filter
rather than a standalone directional trading rule.

### Step 4: Building and Comparing Risk-Managed Trading Strategies

**What this step does:**
This cell constructs two independent directional trading signals — a trend-following momentum
strategy (10-day vs. 50-day moving average crossover) and a short-term mean-reversion strategy
(reversal after a 5-day price move) — applied across all 30 stocks. Each strategy is then run in
two versions: a baseline version, and a version overlaid with a VPIN-based position-sizing rule
that cuts exposure by 70% whenever a stock's VPIN is in its top 20th percentile.

**Why we are doing this:**
Step 3 established that VPIN reliably predicts a volatility spike, but not direction. The logical
next question is whether this predictive power can actually improve a real trading strategy's
risk-adjusted performance — this is what separates a statistically interesting finding from a
practically useful one. Testing two different strategy styles (momentum and mean-reversion) checks
whether VPIN's benefit is a general risk-management effect or something that only works by
coincidence with one particular style of trading.

**Why we scale down position size instead of exiting entirely:**
A full exit would throw away any correct directional signal the base strategy has; a partial
scale-down (70% reduction) reduces risk exposure during dangerous periods while still allowing the
base strategy to participate if its view turns out correct — this is a more realistic and less
aggressive risk control, closer to how real portfolio managers manage volatility risk.

**Why we compare four variants instead of one:**
Running momentum baseline, momentum+VPIN, mean-reversion baseline, and mean-reversion+VPIN side by
side isolates the specific contribution of the VPIN overlay, independent of which directional
strategy is used. If the overlay helps both, that is strong evidence of a general, strategy-agnostic
risk-reduction effect rather than a coincidence tied to one specific strategy.

**What we measure and why:**
Annualized return and volatility give raw performance context. Sharpe ratio measures risk-adjusted
return (return per unit of risk taken) — this is the primary metric funds care about, since a
strategy with lower returns but much lower risk can still be more valuable. Max drawdown measures
the worst peak-to-trough loss, relevant to real-world capital preservation. CVaR at the 5% level
(Conditional Value at Risk) captures the average loss in the worst 5% of days, a tail-risk measure
that Sharpe ratio alone can miss.

**What happens if we skip this step:**
Without this step, VPIN would remain an academic curiosity — a statistically validated but
practically untested signal. A fund evaluating this project would immediately ask "so what do you
actually do with it," and without this step, there would be no answer.

In [6]:
import pandas as pd, numpy as np

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)
ret = close.pct_change()

ma_fast, ma_slow = 10, 50
mom_signal = np.sign(close.rolling(ma_fast).mean() - close.rolling(ma_slow).mean())

rev_window = 5
rev_signal = -np.sign(close.pct_change(rev_window))

vpin_pct = vpin.rank(pct=True)
scale = 1 - 0.7 * (vpin_pct > 0.8).astype(float)

def strat_returns(signal, scale=None):
    pos = signal.shift(1)
    if scale is not None:
        pos = pos * scale.shift(1)
    return (pos * ret).mean(axis=1)

strategies = {
    'momentum_baseline': strat_returns(mom_signal),
    'momentum_vpin_overlay': strat_returns(mom_signal, scale),
    'meanrev_baseline': strat_returns(rev_signal),
    'meanrev_vpin_overlay': strat_returns(rev_signal, scale),
}

perf = pd.DataFrame(strategies).dropna()
perf.to_csv('output/strategy_returns.csv')

def metrics(s):
    ann_ret = s.mean()*252
    ann_vol = s.std()*np.sqrt(252)
    sharpe = ann_ret/ann_vol
    cum = (1+s).cumprod()
    dd = (cum/cum.cummax()-1).min()
    cvar5 = s[s <= s.quantile(0.05)].mean()
    return pd.Series({'AnnRet':ann_ret,'AnnVol':ann_vol,'Sharpe':sharpe,'MaxDD':dd,'CVaR5%':cvar5})

summary = perf.apply(metrics).T
summary.to_csv('output/strategy_summary.csv')
print(summary.round(4))

                       AnnRet  AnnVol  Sharpe   MaxDD  CVaR5%
momentum_baseline      0.0160  0.1324  0.1209 -0.2462 -0.0203
momentum_vpin_overlay  0.0322  0.0927  0.3477 -0.1374 -0.0142
meanrev_baseline       0.0035  0.1351  0.0256 -0.3926 -0.0191
meanrev_vpin_overlay  -0.0082  0.0944 -0.0869 -0.3710 -0.0138


### Step 4 Results: VPIN Overlay Clearly Improves Momentum, Mixed Effect on Mean-Reversion

**What the output shows:**
The momentum strategy improved sharply with the VPIN overlay: Sharpe ratio rose from 0.12 to 0.35,
annualized volatility fell from 13.2% to 9.3%, and max drawdown improved from -24.6% to -13.7%.
The mean-reversion strategy showed a different pattern: the overlay reduced volatility (13.5% to
9.4%) and slightly improved CVaR, but baseline mean-reversion already had almost no edge (Sharpe
0.03), and the overlay pushed its returns slightly negative (Sharpe -0.09).

**Interpreting the numbers:**
Sharpe ratio is the industry-standard measure of return earned per unit of risk; a near-tripling
from 0.12 to 0.35 for momentum is a substantial, meaningful improvement, not a marginal one. The
consistent volatility reduction across both strategies (13%+ down to ~9% in both cases) confirms
VPIN is doing exactly what it was designed to do — flagging and de-risking ahead of volatile
periods — regardless of which directional strategy it's paired with.

**Is this realistic enough to accept:**
Yes, and the asymmetry between the two strategies is itself a credible, non-manufactured result.
If both strategies had improved dramatically, that would raise suspicion of a lucky, overfit result.
Instead, we see the overlay mechanically reduce risk in every case, but only improve overall
risk-adjusted returns when the base strategy has real underlying edge to protect (momentum) — this
is consistent with VPIN's theoretical role as a risk signal, not a return-generation signal.

**How this feeds into the next step:**
The momentum+VPIN result is promising enough to warrant deeper scrutiny before drawing final
conclusions. The next step is a walk-forward validation — testing this result across distinct,
non-overlapping time sub-periods (rather than one full-sample test) — to confirm the improvement
holds out-of-sample and isn't an artifact of a few lucky periods within the 2015-2026 window. This
is a standard institutional-grade check against overfitting before a signal or strategy is
considered "validated."